In [4]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [5]:
# ==================================================
# 1. PREPROCESSING DATA
# ==================================================

df = pd.read_csv('../data/komdigi_hoaks.csv')

df['teks'] = (
    df['title'].fillna('') + ' ' +
    df['excerpt'].fillna('')
)

df = df[
    df['teks'].str.strip() != ''
].reset_index(drop=True)

# Label klasifikasi
df = df.dropna(subset=['topics']).copy()
df['topics'] = df['topics'].astype(str).str.strip()
df = df[df['topics'] != ''].copy()

# Menghapus kelas yang memiliki kurang dari 2 data
counts = df['topics'].value_counts()
df = df[df['topics'].isin(counts[counts >= 2].index)]

print(f"Total data bersih: {len(df)} baris")
print(df[['title', 'teks', 'topics']].head())

X = df['teks']
y = df['topics']

# ==================================================
# 2. PEMBAGIAN DATA
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==================================================
# 3. EKSTRAKSI FITUR TF-IDF
# ==================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ==================================================
# 4. PELATIHAN MODEL
# ==================================================

model = MultinomialNB(alpha=1.0)

start = time.perf_counter()
model.fit(X_train_tfidf, y_train)
training_time = time.perf_counter() - start

# ==================================================
# 5. PREDIKSI
# ==================================================

start = time.perf_counter()
y_pred = model.predict(X_test_tfidf)
prediction_time = time.perf_counter() - start

# ==================================================
# 6. EVALUASI
# ==================================================

print("\n=== MULTINOMIAL NAIVE BAYES ===")
print(f"Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision Macro  : {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall Macro     : {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-Score Macro   : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-Score Weighted: {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Waktu Training   : {training_time:.4f} detik")
print(f"Waktu Prediksi   : {prediction_time:.4f} detik")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ==================================================
# 7. SIMPAN HASIL
# ==================================================

hasil = pd.DataFrame([{
    'Model': 'Multinomial Naive Bayes',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision_Macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
    'Recall_Macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
    'F1_Macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
    'F1_Weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    'Training_Time': training_time,
    'Prediction_Time': prediction_time
}])



Total data bersih: 4176 baris
                                               title  \
0  [HOAKS] Tautan Pencairan BSU September 2026 Se...   
1  [HOAKS] Panas El Nino Berkaitan dengan Gelomba...   
2  [HOAKS] Susu Bebelac Berbahaya karena Mengandu...   
3  [HOAKS] Purbaya Informasikan Raffi Ahmad Beli ...   
4  [HOAKS] Tautan Bantuan Laptop dan Ponsel untuk...   

                                                teks topics  
0  [HOAKS] Tautan Pencairan BSU September 2026 Se...  Hoaks  
1  [HOAKS] Panas El Nino Berkaitan dengan Gelomba...  Hoaks  
2  [HOAKS] Susu Bebelac Berbahaya karena Mengandu...  Hoaks  
3  [HOAKS] Purbaya Informasikan Raffi Ahmad Beli ...  Hoaks  
4  [HOAKS] Tautan Bantuan Laptop dan Ponsel untuk...  Hoaks  

=== MULTINOMIAL NAIVE BAYES ===
Accuracy         : 0.8469
Precision Macro  : 0.1411
Recall Macro     : 0.1667
F1-Score Macro   : 0.1528
F1-Score Weighted: 0.7767
Waktu Training   : 0.0179 detik
Waktu Prediksi   : 0.0010 detik

Classification Report:
          